In [1]:
import pandas as pd
import numpy as np  
from sodapy import Socrata
from datetime import datetime
import os
from tqdm import tqdm
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns


pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# sys.path.append("../src")
sys.path.append(str(Path.cwd().parent))
import utils

path = os.getcwd()
print(path)

path_repo = "C:/Users/PC/Desktop/tesis/"
path_data = "C:/Users/PC/Desktop/tesis/data/"

c:\Users\PC\Desktop\python\rates_degree_work


- La tabla es semanal, terminando en cada viernes

In [ ]:
qwe

## Tasas colocación V1

In [ ]:
tasascol = pd.read_csv(path_repo + "/data/2022.csv", decimal = ",", thousands = ".", encoding='utf-8')
tasascol.columns = [x.lower().replace(" ","_") for x in tasascol.columns]
tasascol.rename(columns = {"tasa_efectiva_promedio_ponderada":"tasa_ponderada", "montos_desembolsados":"monto", "numero_de_creditos_desembolsados":"num_creditos", 
                           "sum_numero_de_creditos":"num_creditos", "producto_de_crédito":"producto", "plazo_de_crédito":"plazo", "nombre_entidad":"banco"}, inplace=True)
tasascol.drop(columns = ["tipo_entidad", "codigo_entidad"], inplace =True)
tasascol.sample(5)

In [ ]:
cols_cat = ['nombre_tipo_entidad', 'banco', 'fecha_corte', 'tipo_de_persona', 'sexo', 'tamaño_de_empresa', 'tipo_de_crédito', 'tipo_de_garantía',
            'producto', 'plazo', 'grupo_etnico', 'antiguedad_de_la_empresa', 'tipo_de_tasa', 'clase_deudor', 'codigo_ciiu', 'codigo_municipio',
            'rango_monto_desembolsado']
cols_cont = ['tasa_ponderada', 'margen_adicional', 'monto', 'num_obl']

tasascol["fecha_corte"] = pd.to_datetime(tasascol["fecha_corte"], dayfirst=True, errors='coerce')

tasascol["tasa_ponderada"] = tasascol["tasa_ponderada"].astype(float)
tasascol["num_creditos"] = tasascol["num_creditos"].astype(int)
tasascol["fecha_corte"] = pd.to_datetime(tasascol["fecha_corte"])
tasascol["anomes"] = tasascol["fecha_corte"].dt.strftime("%Y-%m")

# tasascol = tasascol[~tasascol["banco"].isin(["Fondo Nacional del Ahorro", "La Hipotecaria", "Bancoldex", "Confiar",
#                                         "IRIS C.F.", "Bancamía S.A.", "Bancien", "Mibanco S.A.", "Banco Contactar",
#                                         'Tuya','Crezcamos','Credifamilia','Financiera Juriscoop C.F.','Rappipay',
#                                         'Banco GNB Sudameris','Banco Popular','Coopcentral','Banco Pichincha S.A.','Nu Colombia C.F.',
#                                         "Caja de Vivienda Militar y de Polícia"])] # Excluidos por tasas atipicos
                                        
# tasascol = tasascol[tasascol["tasa_ponderada"] <= 50]
# tasascol = tasascol[tasascol["num_creditos"] >= 2]
# tasascol = tasascol[(tasascol["rango_monto_desembolsado"].notna())] # tienen nulos antes de 202309

tasascol ["banco"] = tasascol["banco"].str.replace("Compañía De Financiamiento","CF").str.replace("Cooperativa Financiera","CF")
tasascol["plazo"] = tasascol["plazo"].replace({'Más de 1 año y hasta 3 años':"(1-3]", 'Más de 3 años y hasta 5 años':"(3-5]",
                                                'Más de 5 años y hasta 7 años':"(5-7]",
                                                'Más de 7 años y hasta 10 años':"(7-10]", 'Más de 10 años':"(10-]",
                                                "Hasta 6 meses":"(0-1]", "Más de 6 meses y hasta 1 año":"(0-1]",
                                                'Hasta 30 días':"(0-1]", 'Entre 31 días y 1 año':"(0-1]"})

tasascol["prom_monto_porobl"] = tasascol["monto"]/tasascol["num_creditos"]

print(tasascol["fecha_corte"].min(), "|", tasascol["fecha_corte"].max())

print(tasascol.shape)
print("Bancos:", tasascol["banco"].nunique())

tasascol.sample(7)

In [ ]:
tasascol[tasascol["anomes"] == "2022-08"]["fecha_corte"].value_counts()

In [ ]:
utils.vc(tasascol["tipo_de_crédito"])

In [ ]:
utils.vc(tasascol["tipo_de_garantía"])

In [ ]:
tasascol.isnull().mean()

In [ ]:
# Sample
tmp = tasascol.drop(columns = ['margen_adicional', 'grupo_etnico', 'antiguedad_de_la_empresa', 'clase_deudor', 'codigo_ciiu',
       'codigo_municipio', 'anomes', 'prom_monto_porobl']).sample(100)
tmp.to_csv(path_data+'sample.csv', index = False)
tmp.sample(5)

In [ ]:
print(np.percentile(tasascol["tasa_ponderada"], 0.05))
print(np.percentile(tasascol["tasa_ponderada"], 2.25))
print(np.percentile(tasascol["tasa_ponderada"], 5))
print(np.percentile(tasascol["tasa_ponderada"], 50))
print(np.percentile(tasascol["tasa_ponderada"], 95))
print(np.percentile(tasascol["tasa_ponderada"], 97.5))
print(np.percentile(tasascol["tasa_ponderada"], 99.5))

In [ ]:
print(tasascol[tasascol["tasa_ponderada"] <= 10].shape)
print(tasascol[tasascol["tasa_ponderada"] >= 40].shape)

In [ ]:
# ax = tasascol[tasascol["fecha_corte"] >= "2025-01-01"]
ax = tasascol
ax = ax.groupby("producto").agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax.columns = ["producto", "monto", "num_creditos", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_creditos"]/ax["num_creditos"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_creditos"]
ax = ax.sort_values("num_creditos", ascending=False)#.head(30)
ax

In [ ]:
# ax = tasascol[tasascol["fecha_corte"] >= "2025-01-01"]
ax = tasascol
ax = ax.groupby("banco").agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax.columns = ["banco", "monto", "num_creditos", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_creditos"]/ax["num_creditos"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_creditos"]
ax = ax.sort_values("num_creditos", ascending=False)#.head(30)
ax

In [ ]:
tasascol["anomes"].value_counts(dropna = False, normalize =True)
ax = tasascol
ax = ax.groupby(["anomes"]).agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax.columns = ["mes", "monto", "num_creditos", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_creditos"]/ax["num_creditos"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_creditos"]
#ax.sort_values("monto", ascending=False).head(40)
ax

In [ ]:
ax = tasascol
ax = ax.groupby(["anomes", "plazo", "rango_desembolso"]).agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":"mean"}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax["monto"] = ax["monto"]/1000000
ax["prom_monto_porobl"] = ax["monto"]/ax["num_creditos"] # millones

#ax.sort_values("monto", ascending=False).head(40)
tasascol = ax
tasascol

## How to group?

In [ ]:
print(tasascol.shape)
tasascolgr = tasascol.copy()
tasascolgr[cols_cat] = tasascolgr[cols_cat].fillna("NULL")
tasascolgr["tasaXmonto"]  = tasascolgr['tasa_ponderada']*tasascolgr['monto']
tasascolgr = tasascolgr.groupby(['nombre_tipo_entidad', 'banco', 'fecha_corte', 'tipo_de_crédito', 'tipo_de_garantía', 'tipo_de_persona', #'sexo', 'tamaño_de_empresa',  'producto',
                                'plazo', #'grupo_etnico', 'antiguedad_de_la_empresa','tipo_de_tasa', 
                                'rango_monto_desembolsado', #'clase_deudor',
                                #'codigo_ciiu', 'codigo_municipio', 'anomes', 'prom_monto_porobl'
                                ]).agg({"monto":"sum", "tasaXmonto":"sum", 'num_creditos':"sum"}).reset_index()
print(tasascolgr.shape)
tasascolgr.sample(5)

In [ ]:
601806204910.0000/30926219000.0000

In [ ]:
qwwe

## Tasas colocación V2

- Tabla semanal que termina en el viernes de cada semana
- H2002-2026

In [ ]:
tasascol = pd.read_csv(path_data+"tasascol_H2002-2026.csv", decimal = ",")
tasascol.columns = [x.lower().replace(" ","_") for x in tasascol.columns.tolist()]
tasascol.rename(columns={'año(aaaa)_-_semana(ss)':"semana", 'fecha_(dd/mm/aaaa)':"fecha", 'nombre_cuenta':"producto", 'clasificación_tipo_establecimiento':"tipo_entidad", 
                         'tasa_(%)':"tasa", 'monto_(miles_de_millones_cop)':"monto"}, inplace =True)
tasascol["fecha"] = pd.to_datetime(tasascol["fecha"], dayfirst = True)
tasascol["anomes"] = tasascol["fecha"].astype(str).str[:7]

print(len(tasascol))
print(min(tasascol["fecha"]))
print(max(tasascol["fecha"]))

tasascol

In [ ]:
tasascol.tail(10)

In [ ]:
ax = tasascol[(tasascol["producto"] == "Tasa de interés de colocación Total") & (tasascol["tipo_entidad"] == "Total establecimientos")]
ax["tasaXMonto"] = ax["tasa"] * ax["monto"]
ax = ax.groupby(['anomes', 'producto', 'tipo_entidad'])[['tasaXMonto', 'monto']].sum().reset_index()
ax["tasa"] = ax["tasaXMonto"] / ax["monto"]
del ax["tasaXMonto"]

In [ ]:
tasascol[(tasascol["producto"] == "Tasa de interés de colocación Total") & (tasascol["tipo_entidad"] == "Total establecimientos")].tail(4)

In [ ]:
# Valid ponderacion
display(tasascol[(tasascol["fecha"] == "2026-10-23") & (tasascol["producto"] == "Crédito de consumo") & (tasascol["tipo_entidad"] == "Total establecimientos")])

ax = tasascol[(tasascol["fecha"] == "2026-10-23") & (tasascol["producto"] == "Crédito de consumo")  & (tasascol["tipo_entidad"] != "Total establecimientos")]
ax["tasaXMonto"] = ax["tasa"] * ax["monto"]
print(ax["monto"].sum())
print((ax["tasaXMonto"].sum()/ax["monto"].sum()))
ax

In [ ]:
# Valid ponderacion
display(tasascol[(tasascol["fecha"] == "2026-10-23") & (tasascol["producto"] == "Tasa de interés de colocación Total")])# & (tasascol["tipo_entidad"] == "")])

ax = tasascol[(tasascol["fecha"] == "2026-10-23") & (~tasascol["producto"].isin(["Tasa de interés de colocación Total", "Tasa de interés de colocación Banco de la República Semanal",
                "Tasa de interés de colocación sin Tesorería"])) & (tasascol["tipo_entidad"] != "Total establecimientos")] #& (tasascol["producto"] == "Crédito de consumo")  & (tasascol["tipo_entidad"] != "Total establecimientos")]
ax["tasaXMonto"] = ax["tasa"] * ax["monto"]
ax = ax.groupby(["semana", "fecha", "tipo_entidad"])[["tasaXMonto", "monto"]].sum().reset_index()
ax["tasa"] = ax["tasaXMonto"] / ax["monto"]

print(ax["monto"].sum())
# print((ax["tasaXMonto"].sum()/ax["monto"].sum()))
ax

In [ ]:
tasascol2 = tasascol.copy()

## Filters

In [ ]:
tasascol = tasascol2.copy()
tasascol["vari_monto_m1"] = tasascol.groupby(["producto", "tipo_entidad"])["monto"].pct_change(1)*100
tasascol["vari_tasa_m1"] = tasascol.groupby(["producto", "tipo_entidad"])["tasa"].pct_change(1)*100
tasascol["monto_r12"] = tasascol.groupby(["producto", "tipo_entidad"])["monto"].shift(12)
tasascol["tasa_r12"] = tasascol.groupby(["producto", "tipo_entidad"])["tasa"].shift(12)
tasascol["y_vari_anual_monto"] = 100*((tasascol["monto"] / tasascol["monto_r12"]) - 1)
tasascol["vari_anual_tasa"] = 100*((tasascol["tasa"] / tasascol["tasa_r12"]) - 1)

tasascol.drop(columns = ["monto_r12", "tasa_r12", "semana", "tipo_entidad"], inplace = True)

print(len(tasascol))
tasascol = tasascol[(tasascol["tipo_entidad"] == "Bancos comerciales")]
tasascol = tasascol[(~tasascol["producto"].isin(["Tasa de interés de colocación Total", "Tasa de interés de colocación Banco de la República Semanal",
                                                 "Tasa de interés de colocación sin Tesorería"]))]
tasascol = tasascol[(tasascol["fecha"] >= "2003-06-01")]
print(len(tasascol))

print(min(tasascol["fecha"]))

tasascol.tail(15)

In [ ]:
ax = tasascol[(tasascol["producto"].isin(["Créditos Comerciales (Ordinario)"]))].tail(200)
ax

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 4))
sns.lineplot(data=ax, x="fecha", y="y_vari_anual_monto", color="b", markersize = 5)
plt.title("", fontsize=14, fontweight="bold")
plt.xlabel("", fontsize=12)
plt.ylabel("", fontsize=12)
plt.xticks(rotation=90, fontsize = 8)
plt.axhline(0, color = "gray", linestyle = "--")
plt.tight_layout()
plt.grid(None)
plt.show()

In [ ]:
for i in tasascol["producto"].unique():
    ax = tasascol[tasascol["producto"] == i].tail(200)

    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(15, 4))
    sns.lineplot(data=ax, x="fecha", y="monto", color="b", markersize = 5)  # y_vari_anual_monto
    plt.title(i, fontsize=14, fontweight="bold")
    plt.xlabel("", fontsize=12)
    plt.ylabel("", fontsize=12)
    plt.xticks(rotation=90, fontsize = 8)
    # plt.axhline(0, color = "gray", linestyle = "--")
    plt.tight_layout()
    plt.grid(None)
    plt.show()

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 4))
sns.lineplot(data=ax, x="fecha", y="vari_monto_m1", color="b", markersize = 5)
plt.title("", fontsize=14, fontweight="bold")
plt.xlabel("", fontsize=12)
plt.ylabel("", fontsize=12)
plt.xticks(rotation=90, fontsize = 8)
plt.axhline(0, color = "gray", linestyle = "--")
plt.tight_layout()
plt.grid(None)
plt.show()

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 4))
sns.lineplot(data=ax, x="fecha", y="monto", color="b", markersize = 5)
plt.title("", fontsize=14, fontweight="bold")
plt.xlabel("", fontsize=12)
plt.ylabel("", fontsize=12)
plt.xticks(rotation=90, fontsize = 8)
# plt.axhline(0, color = "gray", linestyle = "--")
plt.tight_layout()
plt.grid(None)
plt.show()

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 4))
sns.lineplot(data=ax, x="fecha", y="tasa", color="b", markersize = 3)
plt.title("", fontsize=14, fontweight="bold")
plt.xlabel("", fontsize=12)
plt.ylabel("", fontsize=12)
plt.xticks(rotation=90, fontsize = 8)
# plt.axhline(0, color = "gray", linestyle = "--")
plt.tight_layout()
plt.grid(None)
plt.show()

In [ ]:
qwe

In [ ]:
# tasascol.tail(200).to_csv(path_data + "tasascol_sample.csv")

## BANREP

In [ ]:
def calc_viernes(df, columna_fecha='Fecha', saltar_fin_de_semana=False):
    df_resultado = df.copy()
    
    # Asegurar formato datetime
    fechas_dt = pd.to_datetime(df_resultado[columna_fecha])
    dia_semana = fechas_dt.dt.weekday
    
    # Calcular días de diferencia hasta el viernes (Viernes = 4)
    if saltar_fin_de_semana:
        dias_al_viernes = np.where(dia_semana > 4, 11 - dia_semana, 4 - dia_semana)
    else:
        dias_al_viernes = 4 - dia_semana
        
    # Asignar la nueva columna
    df_resultado["Fecha_viernes"] = fechas_dt + pd.to_timedelta(dias_al_viernes, unit='d')
    
    return df_resultado


def filtrar_dias_habiles(df, columna_fecha='Fecha'):
    df_resultado = df.copy()
    
    # Asegurar que la columna tenga formato datetime
    fechas_dt = pd.to_datetime(df_resultado[columna_fecha])
    
    # .dt.weekday devuelve 0=Lunes hasta 6=Domingo. 
    # Los días menores a 5 son de lunes a viernes.
    mascara_habiles = fechas_dt.dt.weekday < 5
    
    # Retornar el DataFrame filtrado
    return df_resultado[mascara_habiles].copy()

In [ ]:
ph = [x for x in os.listdir(path_data+"BANREP/") if "trm" not in x]
print(len(ph))
ph

In [ ]:
banrep = pd.read_excel(path_data+"BANREP/trm.xlsx", decimal = ",")
banrep = banrep.iloc[1:(len(banrep)-1)].dropna(subset = "Fecha")

for i in ph:
    tmp = pd.read_excel(path_data+"BANREP/"+i, decimal = ",")
    tmp = tmp.iloc[1:(len(tmp)-1)].dropna(subset = "Fecha")
    
    banrep["Fecha"] = banrep["Fecha"].astype(str)
    tmp["Fecha"] = tmp["Fecha"].astype(str)
    banrep = pd.merge(banrep, tmp, on = "Fecha", how = "outer")

    print(f"{i}: {tmp.shape}")

print("\n", banrep.shape)
# banrep1.columns = [x.lower().replace("í","i").replace("ó","o").replace("é","e").replace(" ", "_") for x in banrep1.columns]
banrep = banrep.replace({'-':np.nan})
banrep["Fecha"] = pd.to_datetime(banrep["Fecha"], dayfirst = True)
banrep["anomes"] = banrep["Fecha"].dt.strftime("%Y-%m")

banrep = banrep.sort_values("Fecha")

for i in [x for x in banrep if x not in ["Fecha", "anomes"]]:
    try:
        banrep[i] = banrep[i].astype(float)
    except:
        banrep[i] = banrep[i].str.replace(".","").str.replace(",",".").astype(float)

banrep = banrep[banrep["Fecha"] <= max(tasascol["fecha"])]

banrep.columns = [x.replace("Tasa de Certificado de Depósito a Término","Tasa CDT").replace("Tasa de Depósitos a Término Fijo","Tasa CDT").
                   replace("Tasa Representativa del Mercado (TRM)","TRM").replace("Indicador Bancario de Referencia (IBR)","IBR").
                   replace("Índice de precios al consumidor (IPC)","IPC").replace("Tasa de política monetaria","TPM").
                   replace("Captaciones a través de CDT","Tasa CDT") for x in banrep.columns]

dels = [x for x in banrep if "Variación porcentual" in x or "Dato fin de año" in x or "Dato fin de trimestre" in x or "Dato fin de semestre" in x]
banrep.drop(columns = dels, inplace = True)
print("After dels:", banrep.shape)

banrep = calc_viernes(banrep, columna_fecha='Fecha')

banrep = banrep[["Fecha", "Fecha_viernes"] + [x for x in banrep if x not in ["Fecha", "Fecha_viernes"]]]

print("\n", banrep["Fecha"].min(), "|", banrep["Fecha"].max())
print(banrep.shape)
banrep.tail(8)

In [ ]:
banrep_dia = banrep[["Fecha", "Fecha_viernes"] + [x for x in banrep if "Dato diario" in x]]
print(banrep_dia.shape)

banrep_dia = calc_viernes(banrep_dia, columna_fecha='Fecha')
banrep_dia = filtrar_dias_habiles(banrep_dia, columna_fecha='Fecha')

print(banrep_dia.shape)
banrep_dia.tail(8)

In [ ]:
banrep_seman = banrep[["Fecha", "Fecha_viernes"] + [x for x in banrep if "Dato semanal" in x]]
cols = [x for x in banrep_seman if x not in ["Fecha", "Fecha_viernes"]] 
banrep_seman = banrep_seman.dropna(subset=cols, how='all')

banrep_seman = calc_viernes(banrep_seman, columna_fecha='Fecha')
banrep_seman = filtrar_dias_habiles(banrep_seman, columna_fecha='Fecha')

print(banrep_seman.shape)
banrep_seman.tail(8)

In [ ]:
banrep_mes = banrep[["Fecha", "Fecha_viernes"] + [x for x in banrep if "Dato fin de mes" in x]]
cols = [x for x in banrep_mes if x not in ["Fecha", "Fecha_viernes"]] 
banrep_mes = banrep_mes.dropna(subset=cols, how='all')

print(banrep_mes.shape)
banrep_mes.tail(8)

In [ ]:
falta trimestral -> solo pib

In [ ]:
falta anual -> pib, salario minimo

## FED rate

In [ ]:
fedrate = pd.read_csv(path_data + "FED_rate.csv", sep = ",", decimal = ",")
fedrate = fedrate.rename(columns = {"observation_date":"fecha", "FEDFUNDS":"FED rate"})
fedrate["fecha"] = pd.to_datetime(fedrate["fecha"])
print(fedrate.shape)
print(min(fedrate["fecha"]))
print(max(fedrate["fecha"]))
fedrate

## FED

In [ ]:
fed = pd.read_csv(path_data + "FED.csv", sep = ",", decimal = ",")
fed = fed.iloc[5:].rename(columns = {"Series Description":"fecha"}).replace({"ND":np.nan})
fed["fecha"] = pd.to_datetime(fed["fecha"])
cols = [x for x in fed if x not in ["fecha"]]
fed[cols] = fed[cols].astype(float)
print(fed.shape)
print(min(fed["fecha"]))
print(max(fed["fecha"]))
fed.tail(5)

## European Central Bank

In [ ]:
ecb = pd.read_csv(path_data + "ECB.csv", sep = ",")
ecb = ecb.rename(columns = {"DATE":"fecha", "Main refinancing operations - fixed rate tenders (fixed rate) (date of changes) - Level (FM.B.U2.EUR.4F.KR.MRR_FR.LEV)":"ECB rate"})
ecb["fecha"] = pd.to_datetime(ecb["fecha"])
del ecb["TIME PERIOD"]

print(ecb.shape)
print(min(ecb["fecha"]))
print(max(ecb["fecha"]))

ecb.tail(10)

## MSCI Colombia

In [ ]:
esta desde 2012

In [ ]:
msci = pd.read_csv(path_data + "MSCI_Colombia.csv", sep = ",", decimal = ".", thousands=",")
msci = msci.rename(columns = {"Fecha":"fecha", "Último":"MSCI_close", "Apertura":"MSCI_open", "% var.":"vari_MSCI"})
msci["fecha"] = pd.to_datetime(msci["fecha"], dayfirst = True)
msci = msci[["fecha", "MSCI_close", "MSCI_open", "vari_MSCI"]].sort_values("fecha")

for i in ["MSCI_close", "MSCI_open", "vari_MSCI"]:
    msci[i] = msci[i].str.replace(".","").str.replace(",",".").str.replace("%","").astype(float)

print(msci.shape)
print(min(msci["fecha"]))
print(max(msci["fecha"]))

msci.tail(10)

## ISE

In [10]:
ise = pd.read_csv(path_data + "ISE.csv", sep = ";", decimal = ",")
ise["fecha"] = pd.to_datetime(ise["fecha"], dayfirst = True)


print(ise.shape)
print(min(ise["fecha"]))
print(max(ise["fecha"]))

ise["fecha"] = ise["fecha"].astype(str).str[:7]

ise.tail(10)

(260, 3)
2005-01-01 00:00:00
2026-08-01 00:00:00


C:\Users\PC\AppData\Local\Temp\ipykernel_15932\3298215710.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ise["fecha"] = pd.to_datetime(ise["fecha"], dayfirst = True)


,fecha,ISE_original,ISE_ajustado
250,2025-11,131.0000,125.4000
251,2025-12,140.0000,125.9000
252,2026-01,117.5000,126.2000
253,2026-02,119.5000,126.8000
254,2026-03,128.9000,128.3000
255,2026-04,124.2000,128.6000
256,2026-05,129.4000,130.2000
257,2026-06,127.3000,128.7000
258,2026-07,131.7000,129.7000
259,2026-08,127.3000,125.6000


## Yahoo

In [ ]:
import yfinance as yf

dicc = {
# Divisas
"EURUSD=X":"euro_dolar", 
"GBPUSD=X":"libra_dolar", 
"JPY=X":"yen_dolar",
"CNY=X":"yuan_dolar", 
"COP=X": "usd_cop",
"DX-Y.NYB": "dxy_dolar",
# Tasas de Bonos del Tesoro de EE.UU.
"^IRX":"bonos_usa_1y", 
"^FVX":"bonos_usa_5y", 
"^TNX":"bonos_usa_10y", 
"^TYX":"bonos_usa_30y",
# Índices bursátiles globales
"^GSPC": "sp500",
"^IXIC": "nasdaq",
"^DJI": "dow_jones",
"EEM": "msci_emergentes",
"URTH": "msci_world",
# Índices de volatilidad / riesgo
"^VIX": "vix",
# Petróleo
"CL=F": "petrol_wti",
"BZ=F": "petrol_brent",
# Metales
"GC=F": "oro",
"SI=F": "plata",
"HG=F": "cobre",
# Energía
"NG=F": "gas_natural",
# Tasas internacionales / futuros
"SR1=F": "sofr_futuro_1m",
"GE=F": "eurodolar_futuro",
# ETFs de bonos internacionales
"TLT": "bonos_largo_usa",
"SHY": "bonos_corto_usa",
"LQD": "bonos_corps_inversion",
"HYG": "bonos_high_yield",
"EMB": "bonos_mercados_emergentes",
# Criptomonedas (liquidez global)
"BTC-USD": "bitcoin",
"ETH-USD": "ethereum"
}
yahoo = pd.DataFrame(bpmensual["anomes"])

for sym, name in dicc.items():
    #print(name)
    _temp = yf.Ticker(sym)
    _temp = _temp.history(period="10y")
    _temp = _temp.reset_index()
    _temp.rename(columns = {"Date":"fecha", "Close":name}, inplace = True)
    _temp["fecha"] = pd.to_datetime(_temp["fecha"])
    _temp["fecha"] = _temp["fecha"].dt.strftime("%Y-%m")
    _temp = _temp.sort_values("fecha", ascending = False).drop_duplicates("fecha")[["fecha", name]]
    #_temp = _temp.groupby("fecha")[name].mean().reset_index()
    yahoo = pd.merge(yahoo, _temp, on="fecha", how="left")

print(min(yahoo["fecha"]))
print(max(yahoo["fecha"]))

yahoo

## Captaciones

In [ ]:
tasascap = pd.read_csv(path_data + "tasas_de_interes_captacion.csv", decimal = ",")
tasascap.columns = [x.lower() for x in tasascap.columns]
tasascap.rename(columns = {"fechacorte":"fecha"}, inplace = True)
tasascap["fecha"] = pd.to_datetime(tasascap["fecha"], dayfirst = True)

print(tasascap.shape)
print(min(tasascap["fecha"]))
print(max(tasascap["fecha"]))

tasascap.tail(5)

## Cartera vencida

In [ ]:
venci = pd.read_csv(path_data + "cartera_vencida.csv", decimal = ",")
venci.columns = [x.lower() for x in venci.columns]
venci.rename(columns = {"fecha_corte":"fecha"}, inplace = True)
venci["fecha"] = pd.to_datetime(venci["fecha"], dayfirst = True)

print(venci.shape)
print(min(venci["fecha"]))
print(max(venci["fecha"]))

venci.tail(4)

In [ ]:
venci["FECHA_CORTE"].describe()

## Merge

In [ ]:
# Mensual
tasascol["anomes"] = tasascol["anomes"].astype(str)
banrep_mes["anomes"] = banrep_mes["Fecha"].astype(str).str[:7]

print(tasascol.shape)
df = pd.merge(tasascol, banrep_mes.drop(columns = ["Fecha", "Fecha_viernes"]), left_on = "anomes", right_on = "anomes", how = "left", indicator = True)
display(utils.vc(df["_merge"]))
print(df.shape)


# Semanal



# Diario



# Trimestal


# Anual



# FED rate



# FED - Other variables



# Cartera vencida



# Yahoo





df = df.sort_values("fecha")

df.tail(5)

In [ ]:
df.iloc[:len(df)-28].tail(5)

In [ ]:
df.sample(3)

In [ ]:
var = "monto"
cols = [x for x in df if x not in ["fecha", "producto", "anomes", "_merge"]]
ax = df[df["producto"] == "Créditos Comerciales (Ordinario)"][cols]
ax = ax.iloc[:len(ax)-28]
ax

In [ ]:
_cor = ax.corr().reset_index()
_cor = pd.melt(_cor, id_vars = "index")
_cor = _cor[_cor["index"] != _cor["variable"]]
_cor = _cor[_cor["variable"] == var].sort_values("value", ascending = False).drop(columns = "variable")
_cor.head(40)

In [ ]:
_cor.tail(20)

## Feature Enginering

In [ ]:
# Deflactacion de la variabilidad

# Medias moviles del monto 

# Medias moviles de la variabilidad

# Rezagos de la variabilidad

# Rezagos del monto

# Rezagos y medias moviles de la tasa

# Margen de Intermediación (Spread): Diferencia entre las tasas de colocación y captación. Si el spread se comprime, la oferta crediticia tiende a volverse más restrictiva.   
# $$\text{Spread}_t = \text{Tasa Ponderada de Colocación}_t - \text{Tasa Ponderada de Captación}_t$$

# Tasa Usura (Límite Legal): Afecta directamente la colocación en las modalidades de tarjeta de crédito y consumo express.

# Profundización Financiera (Crédito / PIB)Mide la relación estructural entre el tamaño del sistema crediticio y la economía real.
# Fórmula:$$\text{Profundización}_t = \left( \frac{\text{Saldo Cartera Total}_t}{\text{PIB Nominal Trimestral o Anual}_t} \right) X 100$$

# Proporción de cartera vencida sobre cartera total,

# Crecimiento interanual del desembolso real.

# Variación mensual.

# Rezagos de 1, 3, 6 y 12 meses.

# Promedios móviles.

# Diferenciales de tasas.

# Variables de estacionalidad.

# Indicadores de condiciones financieras.

In [ ]:
"""
Primera: incorporar información financiera adicional.

No tienes que limitarte a inflación, desempleo, ISE y TRM. Los spreads de tasas, indicadores bursátiles y condiciones de liquidez también pueden aportar información.

Segunda: construir indicadores derivados.

En vez de incluir únicamente las tasas de interés, puedes construir:
- Diferencial entre tasas activas y tasa de intervención.
- Diferencial entre tasas de captación y colocación.
- Variaciones de tasas.
- Pendiente de la curva de rendimientos.
Tercera: aprovechar los rezagos.

Los autores muestran que los valores actuales y rezagados de los indicadores financieros pueden contener información relevante.

Para tus datos mensuales, probaría rezagos de 1, 3, 6 y 12 meses, seleccionados dentro de la validación temporal.
"""

In [ ]:
# Deflactacion de desembolsos

df["monto_real"] = df["monto"] / df["ipc"] * 100

df["monto_real_lag52"] = df.groupby(
    ["tipo_entidad", "producto"]
)["monto_real"].shift(52)

df["Y_t_real"] = (
    df["monto_real"] / df["monto_real_lag52"]
) - 1

In [ ]:
## Variable respuesta
ax["vari_monto_mes"] = ax.groupby(["producto", "tipo_entidad"])["monto"].pct_change(1)*100
ax["vari_tasa_mes"] = ax.groupby(["producto", "tipo_entidad"])["tasa"].pct_change(1)*100
ax["monto_r12"] = ax.groupby(["producto", "tipo_entidad"])["monto"].shift(12)
ax["tasa_r12"] = ax.groupby(["producto", "tipo_entidad"])["tasa"].shift(12)
ax["y_crec_anual_monto"] = 100*((ax["monto"] / ax["monto_r12"]) - 1)
ax["crec_anual_tasa"] = 100*((ax["tasa"] / ax["tasa_r12"]) - 1)

ax.drop(columns = ["monto_r12", "tasa_r12"], inplace = True)
ax.tail(20)



$ REV
df["fecha"] = pd.to_datetime(df["fecha"])

df = df.sort_values(
    ["tipo_entidad", "producto", "fecha"]
)

# Deflactar desembolsos con el IPC mensual previamente alineado
df["monto_real"] = (df["monto"] / df["ipc"]) * 100

# Desembolsos reales de hace 52 semanas
df["monto_real_lag52"] = df.groupby(
    ["tipo_entidad", "producto"]
)["monto_real"].shift(52) alinear las semanas . años tienen distinto numero de semanas

# Variable respuesta: crecimiento interanual semanal
df["Y_t"] = (
    df["monto_real"] / df["monto_real_lag52"]
) - 1

df["Y_t_pct"] = df["Y_t"] * 100

In [ ]:
df["covid"] = np.where("2020-03-01" <= df["fecha"] <= "2022-06-30", 1, 0)
df["crisis_2008"] = np.where("2008-09-01" <= df["fecha"] <= "2009-10-31", 1, 0)